# AI Video Gen — LTX-Video on Kaggle

Rendering on Kaggle's free GPU quota (**30 hours/week**) instead of a per-clip hosted API.

The pipeline is unchanged: this notebook only supplies the GPU that the `ltx` backend
already expects. Enrichment stays wherever you do it — the enriched prompt is a portable
artefact, which is the whole point of the two-stage split.

## Before you run anything

Two notebook settings, both in the right-hand panel — neither is on by default:

1. **Accelerator → GPU T4 x2** (or P100). Without this every cell runs on CPU and a render
   takes hours instead of a minute.
2. **Internet → On.** Required to clone the repo and pull weights. Kaggle gates this behind
   phone verification on your account; do it once, in Settings.

Quota is billed by *session wall-clock*, not GPU work, so **stop the session when you finish**.
An idle notebook burns the same 30 hours as a busy one.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

import subprocess, sys
out = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if out.returncode != 0:
    sys.exit('No GPU. Set Accelerator to GPU T4 x2 in the notebook settings panel.')


## 1. Fetch the project and LTX-Video

Two separate clones on purpose. LTX-Video's inference dependencies are heavy and pinned;
the project shells out to its CLI rather than importing it, so upstream refactors cannot
break the pipeline and the two dependency sets never have to agree.


In [ ]:
import os, pathlib

WORK = pathlib.Path('/kaggle/working')
PROJECT = WORK / 'Video-Generation-'
LTX = WORK / 'LTX-Video'

# Public repo clones as-is. For a private one, add a Kaggle Secret named GITHUB_TOKEN
# (Add-ons -> Secrets) and this picks it up without putting the token in the notebook.
REPO = 'github.com/PravinderSamra/Video-Generation-'
try:
    from kaggle_secrets import UserSecretsClient
    _tok = UserSecretsClient().get_secret('GITHUB_TOKEN')
    origin = f'https://{_tok}@{REPO}.git'
except Exception:
    origin = f'https://{REPO}.git'

if not PROJECT.exists():
    !git clone --depth 1 {origin} {PROJECT}
if not LTX.exists():
    !git clone --depth 1 https://github.com/Lightricks/LTX-Video.git {LTX}

print('project:', PROJECT.exists(), '| ltx:', LTX.exists())


In [ ]:
# Heavy install: several minutes, and it pulls a matched torch. Expect pip to warn about
# resolver conflicts with Kaggle's preinstalled stack -- harmless, they are for packages
# this pipeline never imports.
!pip install -q -e '{LTX}[inference]' 2>&1 | tail -5
!pip install -q PyYAML 2>&1 | tail -2


## 2. Point the project at LTX

`.env.local` rather than `.env`: it is gitignored and overrides the environment, so a
container-specific path never ends up committed.


In [ ]:
PKG = PROJECT / 'AI Video Gen'   # note the spaces -- quote this path in shell commands
(PKG / '.env.local').write_text(f'LTX_REPO={LTX}\n')
print((PKG / '.env.local').read_text())


## 3. Confirm the backend is ready

`ltx` should read `ready`. If it says `LTX_REPO is not set`, the path above is wrong;
if it names a missing `inference.py`, the clone did not complete.


In [ ]:
%cd "{PKG}"
!python -m src.cli --check


## 4. Pick the variant that fits this card

Be honest about VRAM. On a 16 GB T4 the **2B distilled** model is what fits comfortably.
The 13B distilled weights do not fit in 16 GB without offloading, which is slow enough
to undo the reason you came here.

If you want 13B quality, a rented 24 GB card is the better trade — this notebook is for
free capacity and volume, not for the largest model.


In [ ]:
import torch
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
VARIANT = '13b-distilled' if vram >= 22 else '2b-distilled'
print(f'{vram:.0f} GB detected -> {VARIANT}')


## 5. Render

First run also downloads the weights, so it is much slower than steady state. Start small:
prove the path end to end at 512x320 before scaling resolution or duration.


In [ ]:
!python -m src.cli "a red fox hunting in a snowstorm" \
    --enricher fixture --backend ltx --style nature_doc \
    --seed 1001 --width 512 --height 320 --duration 3


## 6. Look at it, then keep what matters

`/kaggle/working` survives the session and is downloadable from the Output panel; anything
outside it is gone when the session stops.


In [ ]:
import glob, base64
from IPython.display import HTML, display

clips = sorted(glob.glob('outputs/*.mp4'), key=os.path.getmtime)
if not clips:
    print('No clips. Check the render cell output above.')
else:
    latest = clips[-1]
    print(latest, f'{os.path.getsize(latest)/1e6:.1f} MB')
    b64 = base64.b64encode(open(latest, 'rb').read()).decode()
    display(HTML(f'<video controls width=512 src="data:video/mp4;base64,{b64}"></video>'))


In [ ]:
# Sidecars are small, diffable, and the only durable record of how a clip was made.
# Copy them somewhere the session cannot take with it.
!mkdir -p /kaggle/working/keep && cp outputs/*.json /kaggle/working/keep/ 2>/dev/null
!ls -la /kaggle/working/keep/ 2>/dev/null || echo 'no sidecars yet'


## Benchmark set

Ten fixed prompts at fixed seeds. This is the run worth spending quota on — it is what
makes a prompt-template change or a backend swap comparable rather than remembered.
Budget roughly a minute per clip on a T4 at these settings, plus the one-off weight download.


In [ ]:
# Uncomment when a single render above has worked.
# !python -m src.benchmark --backend ltx --enricher fixture
# !python -m src.review outputs/benchmark
